### Tiny Vision Transformer

  tiny vision transformer : for CIFAR-10 dataset
  
  CIFAR-10 dataset
  - 10개의 클래스
  - 32x32 픽셀 이미지
  - 10,000개의 이미지
  - 50,000개의 트레이닝 이미지
  - 10,000개의 테스트 이미지  
  
  Transformer Input 을 위해 이미지를 4x4 패치로 나눔
  - 총 8x8=64개의 패치로 나누어짐
  - 각 패치는 3채널 4x4 픽셀 이미지
  - 각 패치는 4x4x3=48 length vector

  ![CIFAR-10](https://github.com/ultralytics/docs/releases/download/0/cifar10-sample-image.avif)

  Data References:
    1. https://www.cs.toronto.edu/~kriz/cifar.html <br>
    2. https://developer-together.tistory.com/49 <br>
    3. https://tutorials.pytorch.kr/beginner/blitz/cifar10_tutorial.html?highlight=cifar

Packages import

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

print(torch.__version__)

2.7.0+cu126


Torch Device 

In [4]:
if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

print(my_device)

cuda


#### Image to Patch Embedding Vector for VIT

CNN + stride(with patch size) 를 이용하여 Image 를 슬라이딩하며 patch embedding vector 를 만듬

result dim : [batch size, patch size, embedding dim]

In [5]:
class PatchEmbedding(nn.Module):
  # patch_size = 4, in_channels = 3, embed_dim = 48 for CIFAR-10
  def __init__(self, patch_size=4, in_channels=3, embed_dim=48):
    super().__init__()

    # projection : 4x4 3채널 이미지를 1x1 48채널로 변환
    self.projection = nn.Conv2d(
      in_channels=in_channels,
      out_channels=embed_dim,
      kernel_size=patch_size,
      stride=patch_size,
    )

  def forward(self, x):
    # [B, 3, 32, 32] (CIFAR-10 기준, B는 배치 크기) > [B, 48, 8, 8]
    x = self.projection(x)
    # [B, 48, 8, 8] > [B, 48, 64] : [batch size, embedding dim, patch size]
    x = x.flatten(2) # start_dim=2, end_dim=-1


    """
      대부분의 트랜스포머 모델은 입력을 [batch_size, sequence_length, embedding_dim] 형식으로 받음
      따라서 이 형식으로 변환해주는 작업이 필요함
      transpose() : permute 함수와 달리 두 개의 차원만 맞교환 가능능
    """
    # [batch size, embedding dim, patch size] > [batch size, patch size, embedding dim]
    x = x.transpose(1, 2) 
    return x
    

nn.Parameter 

**PyTorch에서 학습 가능한 파라미터(텐서)**를 정의할 때 사용하는 도구입니다. 

일반 Tensor와는 달리, nn.Parameter는 모델의 파라미터로 자동 등록되어 학습 대상이 됩니다.

In [6]:
class XD_TinyVit(nn.Module):
  # mlp_ratio : feed_forward block 의 hidden dim 과 embed_dim 의 비율
  # feed_forward hidden dim = embed_dim * mlp_ratio
  def __init__(self, image_size=32, patch_size=4, in_channels=3, num_classes=10, embed_dim=48, 
               num_heads=4, num_layers=4, dropout=0.1, mlp_ratio=4):
    super().__init__()

    # 이미지를 4x4 패치로 나누어진 패치의 개수
    num_patches = (image_size // patch_size) ** 2

    # 패치 임베딩 레이어 : 4x4 패치를 1x1 48채널로 변환
    self.patch_embedding = PatchEmbedding(patch_size, in_channels, embed_dim)

    # 클래스 토큰 : 패치 임베딩 레이어의 출력을 통해 클래스 토큰을 생성 
    self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
  
    # 위치 임베딩 : 클래스 패치와 패치 임베딩 레이어의 출력을 통해 위치 임베딩을 생성
    self.pos_embedding = nn.Parameter(torch.zeros(1, 1+num_patches, embed_dim))

    # torch transformer encoder 
    # first : torch transformer encoder layer 생성
    encoder_layer = nn.TransformerEncoderLayer(
      d_model=embed_dim,
      nhead=num_heads,
      dim_feedforward=int(embed_dim * mlp_ratio),
      dropout=dropout,
      activation='gelu',
      batch_first=True,
      norm_first=True
    )

    # second : torch transformer encoder 생성
    self.transformer_encoder = nn.TransformerEncoder(
      encoder_layer=encoder_layer,
      num_layers=num_layers
    )


    # last layer normalization
    self.last_lnorm = nn.LayerNorm(embed_dim)

    # classifier layer
    self.classifier = nn.Sequential(
      nn.Linear(embed_dim, embed_dim),
      nn.GELU(),
      nn.Dropout(dropout),
      nn.Linear(embed_dim, num_classes)
    )



  def forward(self, x):
    # patch embedding
    x = self.patch_embedding(x)

    # 클래스 토큰 추가
    cls_token = self.cls_token.expand(x.shape[0], -1, -1)

    # 패치 임베딩과 클래스 토큰 결합
    x = torch.cat([cls_token, x], dim=1)

    # 위치 임베딩 추가
    x = x + self.pos_embedding

    # transformer encoder
    x = self.transformer_encoder(x)

    # last layer normalization
    x = self.last_lnorm(x)

    # 클래스 토큰 추출
    cls_token = x[:, 0, :]

    # 분류 출력
    return self.classifier(cls_token)
  